# DONNEES ARGICOLES 

In [1]:
import pandas as pd

excel_path = "recensement_donnee_agricole.xls"
xls = pd.ExcelFile(excel_path)

all_dfs = []

# Parcourir les 34 feuilles (chacune correspondant à un produit)
for sheet_name in xls.sheet_names:
    df_raw = pd.read_excel(xls, sheet_name=sheet_name, header=None)

    # 1. Repérer la ligne d'en-tête (contient 'COMMUNES' ou 'COMMUNE')
    header_row_idx = None
    for idx, row in df_raw.iterrows():
        row_str = " ".join([str(v) for v in row.values if pd.notna(v)])
        if "COMMUNES" in row_str or "COMMUNE" in row_str:
            header_row_idx = idx
            break

    if header_row_idx is None:
        continue

    row_years = df_raw.iloc[header_row_idx].values
    row_metrics = df_raw.iloc[header_row_idx + 1].values

    # 2. Reconstruire la correspondance des colonnes (Année + Variable)
    cols_info = []
    current_year = None
    for i in range(len(row_years)):
        y = row_years[i]
        m = row_metrics[i]

        if pd.notna(y) and str(y).strip() not in [
            "N°",
            "COMMUNES",
            "COMMUNE",
            "PROD",
        ]:
            current_year = str(y).strip()

        if i == 0:
            cols_info.append(("META", "NO"))
        elif i == 1:
            cols_info.append(("META", "COMMUNE"))
        else:
            m_str = str(m).strip() if pd.notna(m) else ""
            if "SUP" in m_str.upper():
                metric = "superficie"
            elif "REND" in m_str.upper():
                metric = "rendement"
            elif "PROD" in m_str.upper():
                metric = "production"
            else:
                metric = f"unknown_{i}"
            cols_info.append((current_year, metric))

    # 3. Extraction des lignes de communes (exclut les totaux régionaux/nationaux)
    df_data = df_raw.iloc[header_row_idx + 2 :].copy()
    records = []

    for _, row in df_data.iterrows():
        num_val = row.values[0]
        commune_val = row.values[1]

        if pd.notna(commune_val) and pd.notna(num_val):
            try:
                # Vérifie que la ligne correspond bien à un numéro de commune (1 à 77)
                int(num_val)
                commune_name = str(commune_val).strip()

                for col_idx in range(2, len(cols_info)):
                    yr, met = cols_info[col_idx]
                    val = row.values[col_idx]
                    if yr and yr != "None" and "unknown" not in met:
                        records.append({
                            "produit": sheet_name,
                            "communes": commune_name,
                            "annee": yr,
                            "variable": met,
                            "valeur": val,
                        })
            except ValueError:
                continue

    if not records:
        continue

    # 4. Restructuration sous forme de tableau (Produit x Année x Commune)
    df_melted = pd.DataFrame(records)
    df_pivoted = df_melted.pivot_table(
        index=["produit", "annee", "communes"],
        columns="variable",
        values="valeur",
        aggfunc="first",
    ).reset_index()

    all_dfs.append(df_pivoted)

# 5. Fusion globale de toutes les feuilles
df_complet = pd.concat(all_dfs, ignore_index=True)

# Nettoyage et conversion des types numériques
for col in ["production", "rendement", "superficie"]:
    if col in df_complet.columns:
        df_complet[col] = pd.to_numeric(df_complet[col], errors="coerce")

# Réorganisation des colonnes dans l'ordre désiré
df_complet = df_complet[
    ["produit", "annee", "communes", "superficie", "production", "rendement"]
]

print("--- Extraction réussie ---")
print("Nombre total d'observations :", len(df_complet))
print(df_complet.head(10))

# Sauvegarde dans un fichier CSV plat propre
df_complet.to_csv("donnees_agricoles_dataframe.csv", index=False)


--- Extraction réussie ---
Nombre total d'observations : 36574
variable produit      annee         communes  superficie  production  \
0           Maïs  1995-1996           ABOMEY      1317.0      1237.0   
1           Maïs  1995-1996    ABOMEY-CALAVI     21056.0     19519.0   
2           Maïs  1995-1996       ADJA-OUERE     29563.0     30317.0   
3           Maïs  1995-1996          ADJARRA      1705.0      2006.0   
4           Maïs  1995-1996         ADJOHOUN     11396.0     13518.0   
5           Maïs  1995-1996     AGBANGNIZOUN      3677.0      2692.0   
6           Maïs  1995-1996         AGUEGUES       575.0       756.0   
7           Maïs  1995-1996  AKPRO-MISSERETE      3922.0      4458.0   
8           Maïs  1995-1996           ALLADA     17895.0     15891.0   
9           Maïs  1995-1996         APLAHOUE      8926.0      7332.0   

variable    rendement  
0          939.255885  
1          927.004179  
2         1025.504854  
3         1176.539589  
4         1186.205686  


In [2]:
data1 = pd.read_csv("donnees_agricoles_dataframe.csv")
data1.head()

,produit,annee,communes,superficie,production,rendement
0,Maïs,1995-1996,ABOMEY,1317.0,1237.0,939.255885
1,Maïs,1995-1996,ABOMEY-CALAVI,21056.0,19519.0,927.004179
2,Maïs,1995-1996,ADJA-OUERE,29563.0,30317.0,1025.504854
3,Maïs,1995-1996,ADJARRA,1705.0,2006.0,1176.539589
4,Maïs,1995-1996,ADJOHOUN,11396.0,13518.0,1186.205686


### Nettoyage et typpes de données

In [3]:
data1.dtypes

produit           str
annee             str
communes          str
superficie    float64
production    float64
rendement     float64
dtype: object

In [4]:
data1.isna().sum()

produit          0
annee            0
communes         0
superficie      76
production      29
rendement     1986
dtype: int64

### Statistiques des variables numeriques 

In [5]:
data1.describe()

,superficie,production,rendement
count,36498.000000,36545.000000,3.458800e+04
mean,2171.872732,7134.642395,9.076537e+03
std,6882.322427,28898.712565,5.457905e+05
min,0.000000,0.000000,0.000000e+00
25%,24.000000,50.000000,7.878749e+02
50%,197.000000,403.433635,1.609672e+03
75%,1462.000000,2568.000000,6.123911e+03
max,487478.753916,971971.966083,7.300901e+07


In [6]:
data1.shape

(36574, 6)

### Statistiques des variales categorielles 

In [7]:
data1.describe(include =[object])

C:\Users\Masters\AppData\Local\Temp\ipykernel_9820\1680134163.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  data1.describe(include =[object])


,produit,annee,communes
count,36574,36574,36574
unique,34,45,137
top,Maïs,2024-2025,DJOUGOU
freq,2291,2541,608


In [8]:
data1.info()

<class 'pandas.DataFrame'>
RangeIndex: 36574 entries, 0 to 36573
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   produit     36574 non-null  str    
 1   annee       36574 non-null  str    
 2   communes    36574 non-null  str    
 3   superficie  36498 non-null  float64
 4   production  36545 non-null  float64
 5   rendement   34588 non-null  float64
dtypes: float64(3), str(3)
memory usage: 1.7 MB


In [9]:
data1.dropna(inplace= True)
data1.info()

<class 'pandas.DataFrame'>
Index: 34576 entries, 0 to 36573
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   produit     34576 non-null  str    
 1   annee       34576 non-null  str    
 2   communes    34576 non-null  str    
 3   superficie  34576 non-null  float64
 4   production  34576 non-null  float64
 5   rendement   34576 non-null  float64
dtypes: float64(3), str(3)
memory usage: 1.8 MB


In [19]:
data1["produit"].value_counts()

produit
Maïs              2283
Manioc            2271
Arachide          2248
Niébé             2247
Tomate            2193
Piment            2158
Patate douce      2115
Riz               1717
Gombo             1716
Igname            1404
Soja              1340
Sorgho            1219
Voandzou          1189
COTON             1119
Goussi            1094
Anacarde           900
Petit mil          773
Poids d'angole     761
Gboma              672
Crincrin           620
Ananas             508
Taro               482
Laitue             479
Dohi               436
Choux              422
Oignon             414
Carotte            361
Pasteque           273
Sésame             268
Concombre          255
Haricot_vert       223
Fonio              183
Pomme terre        128
Citulus            105
Name: count, dtype: int64

In [ ]:
data1["produit"].value_counts()

# DONNEES METEOROLOGIQUES

In [11]:
import os
import pandas as pd

# Liste de vos fichiers météo
fichiers_meteo = [
    "donne_meteo_2012_2021.xlsx",
    "donne_meteo_2022.xlsx",
    "donne_meteo_2023.xlsx",
    "donne_meteo_2024.xlsx",
    "donne_meteo_2025.xlsx",
]

mois_liste = [
    "Janvier",
    "Février",
    "Mars",
    "Avril",
    "Mai",
    "Juin",
    "Juillet",
    "Août",
    "Septembre",
    "Octobre",
    "Novembre",
    "Décembre",
]


def extraire_donnees_meteo(fichier_path):
    xls = pd.ExcelFile(fichier_path)
    records = []

    for sheet in xls.sheet_names:
        df_raw = pd.read_excel(xls, sheet_name=sheet, header=None)

        # Repérer la ligne d'en-tête (contient COMMUNES ou COMMUNE)
        header_idx = None
        for idx, row in df_raw.iterrows():
            row_str = " ".join([str(v) for v in row.values if pd.notna(v)])
            if "COMMUNE" in row_str.upper():
                header_idx = idx
                break

        if header_idx is None:
            continue

        row_header = df_raw.iloc[header_idx].values

        # Distinguer le fichier multi-années du fichier annuel
        is_multi = "ANNEE" in [
            str(v).upper() for v in row_header if pd.notna(v)
        ]

        # Extraire l'année par défaut à partir du nom de fichier si annuel
        annee_defaut = "".join(
            filter(str.isdigit, fichier_path.replace("2012_2021", ""))
        )

        # Parcourir les lignes de données
        for r_idx in range(header_idx + 2, len(df_raw)):
            row = df_raw.iloc[r_idx].values

            if is_multi:
                commune = row[0]
                annee_val = row[1]
                col_offset = 2
            else:
                num = row[0]
                commune = row[1]
                annee_val = annee_defaut
                col_offset = 2

                # Ne filtrer que les 77 communes (numéro valide de 1 à 77)
                try:
                    int(num)
                except (ValueError, TypeError):
                    continue

            if pd.isna(commune) or str(commune).strip() == "":
                continue

            # Extrait les valeurs pour chacun des 12 mois
            for m_idx, mois_nom in enumerate(mois_liste):
                c_haut = col_offset + (m_idx * 2)
                c_nj = col_offset + (m_idx * 2) + 1

                if c_nj < len(row):
                    haut_val = row[c_haut]
                    nj_val = row[c_nj]

                    records.append({
                        "communes": str(commune).strip(),
                        "annee": int(annee_val),
                        "mois": mois_nom,
                        "num_mois": m_idx + 1,
                        "Haut": haut_val,
                        "N/J": nj_val,
                    })

    return pd.DataFrame(records)


# Extraction et fusion de tous les fichiers
dfs = [extraire_donnees_meteo(f) for f in fichiers_meteo if os.path.exists(f)]
df_meteo_final = pd.concat(dfs, ignore_index=True)

# Nettoyage numérique
df_meteo_final["Haut"] = pd.to_numeric(df_meteo_final["Haut"], errors="coerce")
df_meteo_final["N/J"] = pd.to_numeric(df_meteo_final["N/J"], errors="coerce")

print("--- DataFrame extrait avec succès ---")
print("Nombre total d'enregistrements :", len(df_meteo_final))
print(df_meteo_final.head(12))

# Sauvegarde du fichier intermédiaire
df_meteo_final.to_csv("meteo_combinee_propre.csv", index=False)

--- DataFrame extrait avec succès ---
Nombre total d'enregistrements : 14496
     communes  annee       mois  num_mois    Haut   N/J
0   BEMBEREKE   2012    Janvier         1    0.00   0.0
1   BEMBEREKE   2012    Février         2   45.90   1.0
2   BEMBEREKE   2012       Mars         3    0.00   0.0
3   BEMBEREKE   2012      Avril         4   73.70   3.0
4   BEMBEREKE   2012        Mai         5  107.98   5.0
5   BEMBEREKE   2012       Juin         6  133.25   7.0
6   BEMBEREKE   2012    Juillet         7  316.77  14.0
7   BEMBEREKE   2012       Août         8  136.18  12.0
8   BEMBEREKE   2012  Septembre         9  337.24  16.0
9   BEMBEREKE   2012    Octobre        10   88.97   8.0
10  BEMBEREKE   2012   Novembre        11    0.00   0.0
11  BEMBEREKE   2012   Décembre        12    0.00   0.0


### Affichage du dataset de la meteorologies 

In [12]:
data2 = pd.read_csv("meteo_combinee_propre.csv")
data2.head()

,communes,annee,mois,num_mois,Haut,N/J
0,BEMBEREKE,2012,Janvier,1,0.00,0.0
1,BEMBEREKE,2012,Février,2,45.90,1.0
2,BEMBEREKE,2012,Mars,3,0.00,0.0
3,BEMBEREKE,2012,Avril,4,73.70,3.0
4,BEMBEREKE,2012,Mai,5,107.98,5.0


### Valeurs manquantes ou aberantes  

In [13]:
data2.isna().sum()

communes    0
annee       0
mois        0
num_mois    0
Haut        0
N/J         2
dtype: int64

In [14]:
data2.dropna(inplace = True)


### Statistiques des variables numeriques 

In [15]:
data2.describe()

,annee,num_mois,Haut,N/J
count,14494.000000,14494.000000,14494.000000,14494.000000
mean,2018.284945,6.500483,95.103518,5.312808
std,3.971933,3.452165,123.449446,6.465686
min,2012.000000,1.000000,0.000000,0.000000
25%,2015.000000,4.000000,4.900500,1.000000
50%,2018.000000,7.000000,70.000000,4.833333
75%,2022.000000,9.750000,143.665000,8.000000
max,2025.000000,12.000000,2414.100000,119.950000


### Statistiques des variables categorielle

In [16]:
data2.describe(include = [object])

C:\Users\Masters\AppData\Local\Temp\ipykernel_9820\2881031755.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  data2.describe(include = [object])


,communes,mois
count,14494,14494
unique,90,12
top,BEMBEREKE,Janvier
freq,168,1208


In [17]:
data2["communes"].unique()

<StringArray>
[         'BEMBEREKE',             'KALALE',             'N'DALI',
              'NIKKI',            'PARAKOU',             'PERERE',
            'SINENDE',          'TCHAOUROU',     'DEPART. BORGOU',
          'BANIKOARA',           'GOGOUNOU',              'KANDI',
           'KARIMAMA',         'MALANVILLE',            'SEGBANA',
    'DEPART. ALIBORI',          'BOUKOUMBE',              'COBLY',
              'KEROU',            'KOUANDE',             'MATERI',
         'NATITINGOU',            'PEHUNCO',          'TANGUIETA',
       'TOUCOUNTOUNA',    'DEPART. ATACORA',            'BASSILA',
            'COPARGO',            'DJOUGOU',              'OUAKE',
      'DEPART. DONGA',             'ABOMEY',       'AGBANGNIZOUN',
            'BOHICON',               'COVE',             'DJIDJA',
             'OUINHI',          'ZAGNANADO',           'ZA-KPOTA',
         'ZOGBODOMEY',        'DEPART. ZOU',              'BANTE',
        'DASSA-ZOUME',            'GLAZOUE',    

In [18]:
data2.info()

<class 'pandas.DataFrame'>
Index: 14494 entries, 0 to 14495
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   communes  14494 non-null  str    
 1   annee     14494 non-null  int64  
 2   mois      14494 non-null  str    
 3   num_mois  14494 non-null  int64  
 4   Haut      14494 non-null  float64
 5   N/J       14494 non-null  float64
dtypes: float64(2), int64(2), str(2)
memory usage: 792.6 KB


## **FUSION DES DEUX DATASETS**

In [27]:
import pandas as pd

# 1. Chargement des datasets
df_agri = pd.read_csv('donnees_agricoles_dataframe.csv')
df_meteo = pd.read_csv('meteo_combinee_propre.csv')

# 2. Préparation de la colonne 'annee' dans le dataset agricole (extraire le premier an)
df_agri['annee_debut'] = df_agri['annee'].str.split('-').str[0].astype(int)

# 3. Agrégation annuelle des données météo par commune et année
meteo_annuelle = df_meteo.groupby(['communes', 'annee']).agg({
    'Haut': 'sum',      # Cumul annuel des précipitations
    'N/J': 'sum'        # Nombre total de jours de pluie dans l'année
}).reset_index()

# 4. Fusion (Merge / Join) des deux DataFrames
df_final = pd.merge(
    df_agri, 
    meteo_annuelle, 
    left_on=['communes', 'annee_debut'], 
    right_on=['communes', 'annee'], 
    how='inner'  # Utiliser 'left' pour conserver toutes les lignes agricoles
)

# Nettoyage de la colonne en double
df_final = df_final.drop(columns=['annee_debut', 'annee_y'])

# Affichage du résultat
df_final.head()

,produit,annee_x,communes,superficie,production,rendement,Haut,N/J
0,Maïs,2012-2013,ABOMEY,2205.000000,2168.400000,983.401361,997.10,62.0
1,Maïs,2012-2013,ABOMEY-CALAVI,9916.000000,14314.850000,1443.611335,1186.46,35.0
2,Maïs,2012-2013,ADJA-OUERE,12461.707252,13904.427053,1115.772243,1100.96,47.0
3,Maïs,2012-2013,ADJARRA,1324.033150,1520.088189,1148.074117,1308.30,69.0
4,Maïs,2012-2013,ADJOHOUN,35538.543501,49692.697250,1398.276135,1344.61,66.0


In [28]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 19664 entries, 0 to 19663
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   produit     19664 non-null  str    
 1   annee_x     19664 non-null  str    
 2   communes    19664 non-null  str    
 3   superficie  19616 non-null  float64
 4   production  19651 non-null  float64
 5   rendement   17724 non-null  float64
 6   Haut        19664 non-null  float64
 7   N/J         19664 non-null  float64
dtypes: float64(5), str(3)
memory usage: 1.2 MB
